# Experiment 27 | XGBoost + CatBoost Ensemble Sweep

This experiment tests a large set of diverse XGBoost and CatBoost configurations using the same leakage-safe feature engineering pipeline. The strongest models are then combined using OOF predictions to search for a stronger validation ROC-AUC.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 42
N_SPLITS = 3
SMOOTHING_VALUES = [10.0, 20.0, 40.0]

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

TRAIN_PATH = ROOT / 'data' / 'train.csv'
TEST_PATH = ROOT / 'data' / 'test.csv'
OUTPUT_PATH = ROOT / 'submissions' / 'experiment_27.csv'
RESULTS_PATH = ROOT / 'results' / 'experiment_27_results.csv'

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

TARGET = 'Will_Buy_EV'
y = train[TARGET].map({'No': 0, 'Yes': 1}).astype(np.int8)

X_raw = train.drop(columns=[TARGET]).copy()
X_test_raw = test.copy()

print('Train shape:', X_raw.shape)
print('Test shape:', X_test_raw.shape)


Train shape: (668665, 14)
Test shape: (286571, 14)


In [2]:
categorical_cols = X_raw.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_cols = X_raw.select_dtypes(exclude=['object', 'category']).columns.tolist()

combined = pd.concat([X_raw, X_test_raw], axis=0, ignore_index=True)
combined_encoded = pd.get_dummies(combined, columns=categorical_cols, dtype=np.int8)

X_base = combined_encoded.iloc[:len(X_raw)].copy()
X_test_base = combined_encoded.iloc[len(X_raw):].copy()

print('Base feature count:', X_base.shape[1])


Base feature count: 25


In [3]:
identity_cols = [
    'Age',
    'Annual_Income_USD',
    'Daily_Commute_km',
    'Number_of_Cars_Owned',
    'Charging_Stations_Near_Home',
    'Charging_Stations_Near_Work',
    'Environmental_Concern_Level'
]

pair_cols = [
    ('Age', 'Annual_Income_USD'),
    ('Age', 'Daily_Commute_km'),
    ('Age', 'Current_Car_Type'),
    ('Annual_Income_USD', 'Current_Car_Type'),
    ('Annual_Income_USD', 'City_Type'),
    ('Daily_Commute_km', 'Current_Car_Type'),
    ('Charging_Stations_Near_Home', 'Charging_Stations_Near_Work'),
    ('Environmental_Concern_Level', 'Range_Anxiety_Level')
]

def add_oof_features(X_train_raw, X_valid_raw, y_train, columns, smoothing):
    train_out = pd.DataFrame(index=X_train_raw.index)
    valid_out = pd.DataFrame(index=X_valid_raw.index)

    global_mean = y_train.mean()

    for col in columns:
        stats = pd.DataFrame({
            'key': X_train_raw[col].astype(str),
            'target': y_train.values
        }).groupby('key')['target'].agg(['mean', 'count'])

        smooth = (stats['mean'] * stats['count'] + global_mean * smoothing) / (stats['count'] + smoothing)
        freq = stats['count']

        train_keys = X_train_raw[col].astype(str)
        valid_keys = X_valid_raw[col].astype(str)

        train_out[f'{col}__te'] = train_keys.map(smooth).fillna(global_mean).to_numpy()
        valid_out[f'{col}__te'] = valid_keys.map(smooth).fillna(global_mean).to_numpy()
        train_out[f'{col}__freq'] = train_keys.map(freq).fillna(0).to_numpy()
        valid_out[f'{col}__freq'] = valid_keys.map(freq).fillna(0).to_numpy()

    for cols in pair_cols:
        name = '__'.join(cols)
        train_keys = X_train_raw[list(cols)].astype(str).agg('|'.join, axis=1)
        valid_keys = X_valid_raw[list(cols)].astype(str).agg('|'.join, axis=1)

        stats = pd.DataFrame({
            'key': train_keys,
            'target': y_train.values
        }).groupby('key')['target'].agg(['mean', 'count'])

        smooth = (stats['mean'] * stats['count'] + global_mean * smoothing) / (stats['count'] + smoothing)
        freq = stats['count']

        train_out[f'{name}__te'] = train_keys.map(smooth).fillna(global_mean).to_numpy()
        valid_out[f'{name}__te'] = valid_keys.map(smooth).fillna(global_mean).to_numpy()
        train_out[f'{name}__freq'] = train_keys.map(freq).fillna(0).to_numpy()
        valid_out[f'{name}__freq'] = valid_keys.map(freq).fillna(0).to_numpy()

    return train_out, valid_out


In [4]:
def build_fold_features(train_idx, valid_idx, smoothing):
    X_tr_raw = X_raw.iloc[train_idx]
    X_va_raw = X_raw.iloc[valid_idx]
    y_tr = y.iloc[train_idx]

    extra_tr, extra_va = add_oof_features(
        X_tr_raw,
        X_va_raw,
        y_tr,
        identity_cols,
        smoothing
    )

    X_tr = pd.concat([
        X_base.iloc[train_idx].reset_index(drop=True),
        extra_tr.reset_index(drop=True)
    ], axis=1)

    X_va = pd.concat([
        X_base.iloc[valid_idx].reset_index(drop=True),
        extra_va.reset_index(drop=True)
    ], axis=1)

    return X_tr, X_va

def build_full_features(smoothing):
    extra_train, extra_test = add_oof_features(
        X_raw,
        X_test_raw,
        y,
        identity_cols,
        smoothing
    )

    X_full = pd.concat([
        X_base.reset_index(drop=True),
        extra_train.reset_index(drop=True)
    ], axis=1)

    X_test_full = pd.concat([
        X_test_base.reset_index(drop=True),
        extra_test.reset_index(drop=True)
    ], axis=1)

    return X_full, X_test_full


In [5]:
xgb_configs = [
    {'depth': 4, 'lr': 0.025, 'child': 1, 'sub': 0.90, 'col': 0.85, 'reg': 1.0},
    {'depth': 4, 'lr': 0.035, 'child': 2, 'sub': 0.90, 'col': 0.85, 'reg': 1.0},
    {'depth': 4, 'lr': 0.045, 'child': 2, 'sub': 0.95, 'col': 0.90, 'reg': 1.0},
    {'depth': 4, 'lr': 0.055, 'child': 3, 'sub': 0.90, 'col': 0.90, 'reg': 1.5},
    {'depth': 5, 'lr': 0.025, 'child': 1, 'sub': 0.90, 'col': 0.85, 'reg': 1.0},
    {'depth': 5, 'lr': 0.030, 'child': 2, 'sub': 0.90, 'col': 0.85, 'reg': 1.0},
    {'depth': 5, 'lr': 0.035, 'child': 2, 'sub': 0.90, 'col': 0.85, 'reg': 1.0},
    {'depth': 5, 'lr': 0.040, 'child': 3, 'sub': 0.90, 'col': 0.90, 'reg': 1.0},
    {'depth': 5, 'lr': 0.050, 'child': 3, 'sub': 0.95, 'col': 0.90, 'reg': 1.5},
    {'depth': 5, 'lr': 0.060, 'child': 4, 'sub': 0.90, 'col': 0.95, 'reg': 2.0},
    {'depth': 6, 'lr': 0.025, 'child': 1, 'sub': 0.90, 'col': 0.85, 'reg': 1.0},
    {'depth': 6, 'lr': 0.030, 'child': 2, 'sub': 0.90, 'col': 0.85, 'reg': 1.0},
    {'depth': 6, 'lr': 0.035, 'child': 2, 'sub': 0.85, 'col': 0.85, 'reg': 1.5},
    {'depth': 6, 'lr': 0.045, 'child': 3, 'sub': 0.90, 'col': 0.90, 'reg': 2.0},
    {'depth': 6, 'lr': 0.055, 'child': 4, 'sub': 0.90, 'col': 0.95, 'reg': 2.0},
    {'depth': 7, 'lr': 0.025, 'child': 2, 'sub': 0.85, 'col': 0.85, 'reg': 1.5},
    {'depth': 7, 'lr': 0.035, 'child': 3, 'sub': 0.90, 'col': 0.85, 'reg': 2.0},
    {'depth': 7, 'lr': 0.045, 'child': 4, 'sub': 0.90, 'col': 0.90, 'reg': 2.0},
    {'depth': 8, 'lr': 0.025, 'child': 2, 'sub': 0.85, 'col': 0.80, 'reg': 2.0},
    {'depth': 8, 'lr': 0.035, 'child': 4, 'sub': 0.90, 'col': 0.85, 'reg': 2.5}
]

cat_configs = [
    {'depth': 5, 'lr': 0.035, 'l2': 3},
    {'depth': 6, 'lr': 0.035, 'l2': 3},
    {'depth': 7, 'lr': 0.030, 'l2': 5},
    {'depth': 8, 'lr': 0.025, 'l2': 5},
    {'depth': 6, 'lr': 0.050, 'l2': 5}
]

print('XGBoost configurations:', len(xgb_configs))
print('CatBoost configurations:', len(cat_configs))


XGBoost configurations: 20
CatBoost configurations: 5


In [6]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

model_specs = []

for smoothing in SMOOTHING_VALUES:
    for i, cfg in enumerate(xgb_configs, start=1):
        model_specs.append({
            'name': f'XGB_s{smoothing:g}_{i:02d}',
            'family': 'xgb',
            'smoothing': smoothing,
            'config': cfg
        })

for smoothing in SMOOTHING_VALUES:
    for i, cfg in enumerate(cat_configs, start=1):
        model_specs.append({
            'name': f'CAT_s{smoothing:g}_{i:02d}',
            'family': 'cat',
            'smoothing': smoothing,
            'config': cfg
        })

oof_predictions = {}
model_results = []

for spec in model_specs:
    name = spec['name']
    smoothing = spec['smoothing']
    cfg = spec['config']
    oof = np.zeros(len(X_raw), dtype=np.float64)

    print(f'\\nRunning {name}...')

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X_raw, y), start=1):
        X_tr, X_va = build_fold_features(train_idx, valid_idx, smoothing)
        y_tr = y.iloc[train_idx]
        y_va = y.iloc[valid_idx]

        if spec['family'] == 'xgb':
            model = XGBClassifier(
                n_estimators=1200,
                max_depth=cfg['depth'],
                learning_rate=cfg['lr'],
                min_child_weight=cfg['child'],
                subsample=cfg['sub'],
                colsample_bytree=cfg['col'],
                gamma=0,
                reg_alpha=0,
                reg_lambda=cfg['reg'],
                objective='binary:logistic',
                eval_metric='auc',
                tree_method='hist',
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        else:
            model = CatBoostClassifier(
                iterations=1200,
                depth=cfg['depth'],
                learning_rate=cfg['lr'],
                l2_leaf_reg=cfg['l2'],
                loss_function='Logloss',
                eval_metric='AUC',
                verbose=False,
                random_seed=RANDOM_STATE,
                thread_count=-1
            )

        model.fit(X_tr, y_tr)
        oof[valid_idx] = model.predict_proba(X_va)[:, 1]

    score = roc_auc_score(y, oof)
    oof_predictions[name] = oof
    model_results.append({
        'model': name,
        'family': spec['family'],
        'smoothing': smoothing,
        'validation_auc': score
    })

    print(f'{name} | ROC-AUC: {score:.6f}')

results_df = pd.DataFrame(model_results).sort_values('validation_auc', ascending=False).reset_index(drop=True)
print('\\nTop models:')
print(results_df.head(15).to_string(index=False))


\nRunning XGB_s10_01...
XGB_s10_01 | ROC-AUC: 0.737420
\nRunning XGB_s10_02...
XGB_s10_02 | ROC-AUC: 0.730464
\nRunning XGB_s10_03...


KeyboardInterrupt: 

In [ ]:
top_names = results_df.head(12)['model'].tolist()

blend_results = []
best_blend = None
best_blend_score = -np.inf

def evaluate_blend(names, weights):
    pred = np.zeros(len(y), dtype=np.float64)
    total = sum(weights)
    for name, weight in zip(names, weights):
        pred += oof_predictions[name] * weight
    pred /= total
    return roc_auc_score(y, pred), pred

for a, b in combinations(top_names, 2):
    for wa in range(1, 10):
        wb = 10 - wa
        score, pred = evaluate_blend([a, b], [wa, wb])
        blend_results.append({
            'models': f'{a} + {b}',
            'weights': f'{wa}:{wb}',
            'validation_auc': score
        })
        if score > best_blend_score:
            best_blend_score = score
            best_blend = ([a, b], [wa, wb], pred)

for combo_size in [3, 4, 5]:
    for names in combinations(top_names[:8], combo_size):
        score, pred = evaluate_blend(names, [1] * combo_size)
        blend_results.append({
            'models': ' + '.join(names),
            'weights': ':'.join(['1'] * combo_size),
            'validation_auc': score
        })
        if score > best_blend_score:
            best_blend_score = score
            best_blend = (list(names), [1] * combo_size, pred)

blend_df = pd.DataFrame(blend_results).sort_values('validation_auc', ascending=False).reset_index(drop=True)

print('\\nTop individual models:')
print(results_df.head(10).to_string(index=False))

print('\\nTop blends:')
print(blend_df.head(15).to_string(index=False))

print(f'\\nBest individual ROC-AUC: {results_df.iloc[0].validation_auc:.6f}')
print(f'Best blend ROC-AUC: {best_blend_score:.6f}')
print('Best blend:', best_blend[0])
print('Blend weights:', best_blend[1])


In [ ]:
selected_names, selected_weights, _ = best_blend

test_predictions = []
total_weight = sum(selected_weights)

for name, weight in zip(selected_names, selected_weights):
    spec = next(s for s in model_specs if s['name'] == name)
    smoothing = spec['smoothing']
    cfg = spec['config']

    X_full, X_test_full = build_full_features(smoothing)

    print(f'Training final model: {name}')

    if spec['family'] == 'xgb':
        model = XGBClassifier(
            n_estimators=1200,
            max_depth=cfg['depth'],
            learning_rate=cfg['lr'],
            min_child_weight=cfg['child'],
            subsample=cfg['sub'],
            colsample_bytree=cfg['col'],
            gamma=0,
            reg_alpha=0,
            reg_lambda=cfg['reg'],
            objective='binary:logistic',
            eval_metric='auc',
            tree_method='hist',
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    else:
        model = CatBoostClassifier(
            iterations=1200,
            depth=cfg['depth'],
            learning_rate=cfg['lr'],
            l2_leaf_reg=cfg['l2'],
            loss_function='Logloss',
            eval_metric='AUC',
            verbose=False,
            random_seed=RANDOM_STATE,
            thread_count=-1
        )

    model.fit(X_full, y)
    test_predictions.append(weight * model.predict_proba(X_test_full)[:, 1])

final_prediction = np.sum(test_predictions, axis=0) / total_weight

submission = pd.DataFrame({
    'Will_Buy_EV': final_prediction
})

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(RESULTS_PATH, index=False)

print('\\nSaved:', OUTPUT_PATH)
print('Saved:', RESULTS_PATH)
print('Submission shape:', submission.shape)
print('Prediction range:', final_prediction.min(), 'to', final_prediction.max())


## Experiment 27 result

The final result should be recorded here after the notebook finishes.

**Best individual validation ROC-AUC:** see `results_df` above.

**Best blend validation ROC-AUC:** see `best_blend_score` above.
